# Single-frame ResNet50 steering (spatial 9728 features)

Run the cells in order from a checkout containing the eight native drives under `data/raw/`. The first run needs the pretrained MyDeZero ResNet50 weights. Each execution creates its own dataset manifest, checkpoint and evaluation files. Inference uses only the current RGB image. The primary endpoint is frame-weighted MAE against unmodified PID commands, compared with zero steering on exactly the same frames.

In [ ]:
import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.models import ResNet50, VGG16


class SpatialSteeringResNet(ResNet50):
    """Predict normalized steering from mean-centered BGR frames (N, 3, 120, 160)."""

    def __init__(self, pretrained=False):
        super().__init__(pretrained=pretrained)
        self.fc6 = Linear(1, in_size=512 * 3 * 5 + 2048)

    def frozen_features(self, images):
        with test_mode(), no_grad():
            x = F.pooling(F.relu(self.bn1(self.conv1(images))), 3, stride=2)
            mid = self.res3(self.res2(x))
            top = self.res5(self.res4(mid))
            local = mid.data.reshape(len(images), 512, 3, 5, 5, 4).mean(axis=(3, 5))
            global_features = top.data.mean(axis=(2, 3))
            xp = cuda.get_array_module(global_features)
            return xp.concatenate((local.reshape(len(images), -1), global_features), axis=1)

    def forward(self, images):
        return F.clip(self.fc6(self.frozen_features(images)), -1, 1)

    def steer(self, rgb_frame):
        """Predict one steering command from a 120 by 160 RGB camera frame."""
        frame = np.asarray(rgb_frame)
        if frame.shape != (120, 160, 3) or frame.dtype != np.uint8:
            raise ValueError("Expected a uint8 RGB camera frame of shape (120, 160, 3)")
        pixels = VGG16.preprocess(Image.fromarray(frame), size=None)
        xp = cuda.get_array_module(self.fc6.W.data)
        with test_mode(), no_grad():
            return float(self(xp.asarray(np.ascontiguousarray(pixels[None]))).data.item())

def load_steering_model(checkpoint, use_gpu=True):
    model = SpatialSteeringResNet()
    model.load_weights(checkpoint)
    if model.fc6.W.shape != (9728, 1):
        raise ValueError("Expected a 9728-to-1 steering head")
    if use_gpu and cuda.gpu_enable and cuda.cupy.is_available():
        model.to_gpu()
    return model

In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import json
from pathlib import Path
from uuid import uuid4

import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.models import VGG16
from mydezero.optimizers import Adam
from donkeycar_app.dataset import steering_loss_weights

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/mydezero").is_dir())
seed = 42
np.random.seed(seed)
use_gpu = cuda.gpu_enable and cuda.cupy.cuda.runtime.getDeviceCount() > 0
xp = cuda.cupy if use_gpu else np
run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
out = root / "runs/donkeycar" / run
weights_dir = root / "models/donkeycar" / run
index_dir = root / "data/processed/donkeycar/pid-spike-repair" / run
for folder in (out, weights_dir, index_dir):
    folder.mkdir(parents=True)
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")

print(run, "CUDA" if use_gpu else "CPU")

## Construct the dataset from native recordings

Keep complete drives together. Record the original action and its source. Only training rows can receive a replacement target or a reduced weight. Flag an isolated command at the exact steering limit when both neighbors point in the opposite direction, all three record IDs are consecutive, and all three throttles are positive. Adjacent flagged commands form an oscillation rather than a single impulse; keep them flagged for weighting, but interpolate only when both neighbors are unflagged. Interpolation is a training hypothesis, not a recovered ground truth.

In [ ]:
splits = {"fit": range(1, 6), "validation": [6], "test": [7, 8]}
def read_drive(number, split):
    route = f"trace{number}"
    folder = root / "data/raw" / route
    records = sorted(folder.glob("record_*.json"), key=lambda p: int(p.stem.split("_")[-1]))
    if not records:
        raise FileNotFoundError(folder)
    rows = []
    for path in records:
        raw = json.loads(path.read_text(encoding="utf-8-sig"))
        frame = folder / raw["cam/image_array"]
        angle, throttle = float(raw["user/angle"]), float(raw["user/throttle"])
        if not (frame.is_file() and -1 <= angle <= 1 and np.isfinite(throttle)):
            raise ValueError(path)
        rows.append(dict(route=route, frame=int(path.stem.split("_")[-1]),
                         image=frame.relative_to(root).as_posix(),
                         record=path.relative_to(root).as_posix(), throttle=throttle,
                         raw=angle, target=angle, weight=1.0, spike=False, sustained=False))
    for before, current, after in zip(rows, rows[1:], rows[2:]):
        if (before["frame"] + 1 != current["frame"] or current["frame"] + 1 != after["frame"]
                or min(before["throttle"], current["throttle"], after["throttle"]) <= 0):
            continue
        a, b, c = (r["raw"] for r in (before, current, after))
        current["spike"] = abs(b) == 1 and a*b < 0 and b*c < 0
        current["sustained"] = min(abs(a), abs(b), abs(c)) >= 0.3 and a*b > 0 and b*c > 0
    if split == "fit":
        for before, current, after in zip(rows, rows[1:], rows[2:]):
            if current["spike"]:
                current["weight"] = 0.1
                if not before["spike"] and not after["spike"]:
                    current["target"] = (before["raw"] + after["raw"]) / 2
    return rows

rows = {name: [r for n in drives for r in read_drive(n, name)] for name, drives in splits.items()}
for name, part in rows.items():
    save_json(index_dir / f"{name}.json", part)
    print(name, len(part), "frames;", sum(r["target"] != r["raw"] for r in part), "repaired")
    if name != "fit":
        assert all(r["weight"] == 1 and r["target"] == r["raw"] for r in part)

# Reject exact image overlap between splits, including identical files stored under different paths.
digests = {name: {hashlib.sha256((root / r["image"]).read_bytes()).digest() for r in part}
           for name, part in rows.items()}
assert digests["fit"].isdisjoint(digests["validation"] | digests["test"])
assert digests["validation"].isdisjoint(digests["test"])

summary = []
for split, part in rows.items():
    for route in sorted({r["route"] for r in part}):
        drive = [r for r in part if r["route"] == route]
        pairs = [(a, b) for a, b in zip(drive, drive[1:]) if a["frame"] + 1 == b["frame"]
                 and min(a["throttle"], b["throttle"]) > 0]
        def variation(key):
            return float(np.mean([abs(a[key] - b[key]) for a, b in pairs])) if pairs else None
        summary.append(dict(split=split, route=route, frames=len(drive),
                            spikes=sum(r["spike"] for r in drive),
                            repaired=sum(r["raw"] != r["target"] for r in drive),
                            moving_pairs=len(pairs), raw_mean_abs_delta=variation("raw"),
                            target_mean_abs_delta=variation("target")))
save_json(index_dir / "summary.json", summary)
(index_dir / "README.md").write_text(
    "# PID steering dataset\n\n"
    "Each JSON index lists consecutive records grouped by drive. Image and record paths "
    "are relative to the repository root. The raw images are shared across recipes.\n\n"
    "Fit uses trace1–5, validation uses trace6, and test uses trace7–8. "
    "Use raw for original steering, target for the interpolation recipe, and weight "
    "with raw for the downweighted recipe. The spike and sustained flags mark metric subsets. "
    "Validation and test keep all original steering values and unit weights. "
    "summary.json reports mean absolute steering change per consecutive moving pair.\n",
    encoding="utf-8")

## Extract frozen features

Pool the third residual stage on a 3 × 5 grid to preserve the position of road markings. Append global features from the final stage. Standardization statistics are fitted only on training data. The backbone uses pretrained weights in evaluation mode.

In [ ]:
model = SpatialSteeringResNet(pretrained=True)
if use_gpu:
    model.to_gpu()
def pixels(row):
    with Image.open(root / row["image"]) as im:
        if im.mode != "RGB" or im.size != (160, 120):
            raise ValueError(row["image"])
        return VGG16.preprocess(im, size=None)

def features(part, model):
    result = []
    for start in range(0, len(part), 32):
        batch = np.ascontiguousarray(np.stack([pixels(r) for r in part[start:start + 32]]))
        result.append(cuda.as_numpy(model.frozen_features(xp.asarray(batch))))
    return xp.asarray(np.concatenate(result))

def column(part, key):
    return xp.asarray([r[key] for r in part], dtype=xp.float32).reshape(-1, 1)

fit, val = rows["fit"], rows["validation"]
X = features(fit, model)
V = features(val, model)
center = X.mean(axis=0)
scale = xp.maximum(X.std(axis=0), 0.1)
X -= center
X /= scale
V -= center
V /= scale
print(X.shape, V.shape)

## Compare steering loss weights

Use a smooth absolute-error loss with transition scale 0.05. All heads start at zero and use the same minibatch sequence and learning rate schedule. Retain the raw, spike-downweighted and interpolated controls. Add 2x and 4x weights for original commands with absolute magnitude at least 0.3, excluding flagged reverse spikes. Large commands are a proxy for sharp turns. The multiplier applies only to training weights, and each batch loss is divided by its weight sum.

Evaluate every epoch against original validation labels with unit weights. Track overall MAE, large commands without flagged spikes, ordinary commands and flagged spikes. Save each recipe's best checkpoint by overall validation MAE and export the overall winner as best.npz. Compare subgroup errors before choosing a candidate for closed-loop driving.


In [ ]:
val_y = column(val, "raw")
epochs, batch_size, delta = 24, 256, 0.05
sharp_threshold = 0.3
recipes = {"raw": 1.0, "downweighted": 1.0, "interpolated": 1.0,
           "sharp_x2": 2.0, "sharp_x4": 4.0}
val_spikes = np.array([r["spike"] for r in val], dtype=bool)
val_large = np.abs(cuda.as_numpy(val_y).ravel()) >= sharp_threshold
val_groups = {
    "sharp_nonspike": xp.asarray(val_large & ~val_spikes),
    "ordinary": xp.asarray(~val_large),
    "spikes": xp.asarray(val_spikes),
}
save_json(out / "loss-config.json", dict(sharp_threshold=sharp_threshold,
          multipliers=recipes, spike_weight=0.1, normalization="batch_weight_sum",
          selection="overall_validation_mae"))
history = []
selections = {}
best_mae = float("inf")
for recipe, sharp_multiplier in recipes.items():
    rng = np.random.default_rng(seed)
    head = Linear(1, in_size=X.shape[1])
    head.W.data.fill(0)
    if use_gpu:
        head.to_gpu()
    optim = Adam(lr=0.00008).setup(head)
    y = column(fit, "target" if recipe == "interpolated" else "raw")
    w = xp.ones_like(y)
    if recipe == "downweighted" or sharp_multiplier > 1:
        w = xp.asarray(steering_loss_weights(fit, sharp_multiplier, sharp_threshold))
    for epoch in range(1, epochs + 1):
        optim.lr = 0.00008 * (0.5 ** ((epoch - 1) // 8))
        order = rng.permutation(len(fit))
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            residual = head(X[idx]) - y[idx]
            loss = F.sum(((residual ** 2 + delta ** 2) ** 0.5 - delta) * w[idx]) / float(w[idx].sum())
            head.cleargrads()
            loss.backward()
            optim.update()
        with no_grad():
            pred = xp.clip(head(V).data, -1, 1)
            mae = float(xp.abs(pred - val_y).mean())
        group_mae = {name: float(xp.abs(pred - val_y).ravel()[mask].mean())
                     for name, mask in val_groups.items() if bool(mask.any())}
        history.append(dict(recipe=recipe, epoch=epoch, validation_mae=mae, **group_mae))
        if recipe not in selections or mae < selections[recipe]["mae"]:
            candidate_path = weights_dir / f"{recipe}.npz"
            model.fc6.W.data = head.W.data / scale[:, None]
            model.fc6.b.data = head.b.data - center @ model.fc6.W.data
            model.save_weights(candidate_path)
            selections[recipe] = dict(epoch=epoch, mae=mae, **group_mae,
                                      checkpoint=candidate_path.relative_to(root).as_posix())
        if mae < best_mae:
            best_mae, best_recipe, best_epoch = mae, recipe, epoch
            best_w, best_b = head.W.data.copy(), head.b.data.copy()
    print(recipe, selections[recipe], flush=True)
save_json(out / "history.json", history)
val_baseline = float(xp.abs(val_y).mean())
print("Selected", best_recipe, best_epoch, best_mae, "zero", val_baseline)
if not best_mae < val_baseline:
    raise RuntimeError("No candidate beat zero steering on validation; revisit training before testing.")

## Export and check the selected model

Fold training-set standardization into the regression weights. The checkpoint contains the full backbone and readout. Validate single-image inference after reloading on CPU against the original validation predictions.

In [ ]:
reference = cuda.as_numpy(xp.clip(V[:4] @ best_w + best_b, -1, 1)).ravel()
model.fc6.W.data = best_w / scale[:, None]
model.fc6.b.data = best_b - center @ model.fc6.W.data
checkpoint = weights_dir / "best.npz"
model.save_weights(checkpoint)
inference = SpatialSteeringResNet()
inference.load_weights(checkpoint)
def rgb(row):
    with Image.open(root / row["image"]) as im:
        return np.array(im)
actual = [inference.steer(rgb(r)) for r in val[:4]]
np.testing.assert_allclose(actual, reference, atol=3e-5, rtol=1e-4)
if use_gpu:
    inference.to_gpu()
print("CPU checkpoint round trip passed", checkpoint.relative_to(root))

## Evaluate on the original held-out drives

Use all frames, including spikes. Report frame-weighted errors on the same PID targets for both predictors. The sustained-turn subset requires three consecutive commands with the same sign and magnitude at least 0.3. These drives were used in previous project evaluations. They are a historical holdout, not a fresh blind test.

In [ ]:
def scores(part, predictions):
    targets = np.array([r["raw"] for r in part])
    err = np.abs(predictions - targets)
    zero = np.abs(targets)
    groups = {"all": np.ones(len(part), dtype=bool),
              "sharp": zero >= 0.3,
              "sustained": np.array([r["sustained"] for r in part]),
              "spikes": np.array([r["spike"] for r in part]),
              "sharp_nonspike": (zero >= sharp_threshold) & ~np.array([r["spike"] for r in part]),
              "ordinary": zero < sharp_threshold}
    for route in sorted({r["route"] for r in part}):
        groups[route] = np.array([r["route"] == route for r in part])
    return {k: dict(frames=int(mask.sum()), mae=float(err[mask].mean()),
                    zero_mae=float(zero[mask].mean())) for k, mask in groups.items() if mask.any()}

report = dict(seed=seed, run=run, checkpoint=checkpoint.relative_to(root).as_posix(),
              validation_candidates=selections, selected_recipe=best_recipe, epoch=best_epoch,
              index=index_dir.relative_to(root).as_posix(), device="CUDA" if use_gpu else "CPU")
for name in ("validation", "test"):
    part = rows[name]
    with no_grad(), test_mode():
        if name == "validation":
            pred = cuda.as_numpy(xp.clip(V @ best_w + best_b, -1, 1)).ravel()
        else:
            pred = cuda.as_numpy(F.clip(inference.fc6(features(part, inference)), -1, 1)).ravel()
    assert np.isfinite(pred).all() and len(pred) == len(part)
    report[name] = scores(part, pred)
    with (out / f"{name}-predictions.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(("route", "frame", "target", "prediction"))
        writer.writerows((r["route"], r["frame"], r["raw"], float(p)) for r, p in zip(part, pred))
save_json(out / "metrics.json", report)
print(json.dumps(report, indent=2))

## Load the model for one camera frame

On another machine, install the project, copy the checkpoint, construct `SpatialSteeringResNet()`, and call `load_weights(path)` before `steer(rgb_frame)`. The input is an RGB uint8 array of shape (120, 160, 3). The output is a steering command in [−1, 1].

## Inspect the steering labels and training curves

In [ ]:
import matplotlib.pyplot as plt

summary_fit = json.loads((index_dir / "fit.json").read_text(encoding="utf-8"))
segment = [r for r in summary_fit if r["route"] == "trace3" and 518 <= r["frame"] <= 540]
fig, (trace_ax, score_ax) = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
for key, label in (("raw", "Recorded"), ("target", "Interpolated")):
    trace_ax.plot([r["frame"] for r in segment], [r[key] for r in segment], marker=".", label=label)
trace_ax.set(title="Training labels near an isolated spike", xlabel="Frame in trace3", ylabel="Steering command")
trace_ax.legend()
for recipe in recipes:
    label = recipe
    curve = [r for r in history if r["recipe"] == recipe]
    score_ax.plot([r["epoch"] for r in curve], [r["validation_mae"] for r in curve], label=label)
score_ax.axhline(val_baseline, linestyle="--", color="gray", label="Zero steering")
score_ax.set(title="Performance on trace6", xlabel="Epoch", ylabel="Validation MAE")
score_ax.legend()
fig.savefig(out / "experiment.png", dpi=150)
plt.show()

## Drive a saved checkpoint

Start DonkeySim, run the model definition cell above, then run the following in a cell with your checkpoint path.

```python
from donkeycar_app.drive import run_drive
model = load_steering_model("path/to/checkpoint.npz")
run_drive(model.steer)
```